# 11 · Comparación y Selección de Candidatos — Sprint 3
**Proyecto:** Detección de Sitios Web Fraudulentos (Phishing)  
**Sprint 3 — PB-12:** Comparar modelos, seleccionar top candidatos y documentar decisiones

---

## Objetivo

Consolidar los resultados de todos los baselines en una tabla comparativa,
rankearlos por la métrica principal (F1-Score), analizar trade-offs y
seleccionar los **top 2–3 candidatos** que pasarán al Sprint 4 para tuning.

> **Criterio de selección:** F1-Score (CV) como métrica principal. Se consideran también
> AUC-ROC (discriminación) y Recall (costo del falso negativo en phishing).
> Se penalizan modelos con alto overfitting gap o rendimiento pobre.


## 1. Imports y carga de resultados

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from src.models import get_baseline_models, load_model

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TARGET_COL   = 'Result'

print('✓ Librerías cargadas.')

In [ ]:
# Cargar tabla de resultados generada en 09_baseline_models.ipynb
df_results = pd.read_csv('../models/baseline_comparison.csv')
df_results = df_results.sort_values('F1', ascending=False).reset_index(drop=True)
df_results.index += 1

print(f'Modelos evaluados: {len(df_results)}')
df_results

## 2. Tabla comparativa con resaltado

Verde = mejor valor por métrica | Rojo = peor valor

In [ ]:
display_cols = ['Modelo', 'F1', 'F1_std', 'AUC-ROC', 'Recall', 'Precision', 'Accuracy', 'Overfit_Gap', 'Overfitting?']
metric_cols  = ['F1', 'AUC-ROC', 'Recall', 'Precision', 'Accuracy']

df_results[display_cols].style \
    .highlight_max(subset=metric_cols, color='#c8e6c9') \
    .highlight_min(subset=metric_cols, color='#ffcdd2') \
    .bar(subset=['Overfit_Gap'], color='#ef9a9a', vmin=0, vmax=0.3) \
    .format({col: '{:.4f}' for col in metric_cols + ['F1_std', 'Overfit_Gap']})

## 3. Visualización comparativa — Barras agrupadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- F1, AUC-ROC, Recall por modelo ---
metrics_plot = ['F1', 'AUC-ROC', 'Recall']
x = np.arange(len(df_results))
width = 0.25
colors = ['#1565c0', '#2e7d32', '#b71c1c']

ax0 = axes[0]
for i, (metric, color) in enumerate(zip(metrics_plot, colors)):
    ax0.bar(x + i * width, df_results[metric], width, label=metric,
            color=color, alpha=0.8)

ax0.set_xticks(x + width)
ax0.set_xticklabels(df_results['Modelo'], rotation=35, ha='right', fontsize=8)
ax0.set_ylim(0, 1.05)
ax0.set_title('F1, AUC-ROC y Recall por modelo', fontweight='bold')
ax0.legend()
ax0.axhline(0.80, color='gray', linestyle='--', alpha=0.4)

# --- Overfit gap ---
ax1 = axes[1]
bar_colors = ['#c62828' if g > 0.10 else '#1b5e20' for g in df_results['Overfit_Gap']]
ax1.barh(df_results['Modelo'], df_results['Overfit_Gap'], color=bar_colors, alpha=0.85)
ax1.axvline(0.10, color='orange', linestyle='--', linewidth=1.5, label='Umbral overfitting (0.10)')
ax1.set_xlabel('Gap Train F1 − Test F1')
ax1.set_title('Detección de Overfitting', fontweight='bold')
ax1.invert_yaxis()
ax1.legend(fontsize=8)

plt.suptitle('Comparación de Baselines — Sprint 3', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../models/model_comparison_charts.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Radar Chart — Top 3 modelos

In [ ]:
top3 = df_results.head(3)
metrics_radar = ['F1', 'AUC-ROC', 'Recall', 'Precision', 'Accuracy']
N = len(metrics_radar)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors_radar = ['#1565c0', '#2e7d32', '#c62828']

for i, (_, row) in enumerate(top3.iterrows()):
    values = [row[m] for m in metrics_radar]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, color=colors_radar[i], label=row['Modelo'])
    ax.fill(angles, values, alpha=0.08, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_radar, fontsize=10)
ax.set_ylim(0, 1)
ax.set_title('Radar Chart — Top 3 Baselines', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig('../models/radar_top3.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Selección de candidatos para Sprint 4

### Criterios de selección

1. **F1-Score (CV) ≥ umbral** — métrica principal (mínimo recomendado: 0.80).
2. **AUC-ROC alto** — buena discriminación general.
3. **Overfit_Gap ≤ 0.10** — modelo generaliza bien sin parámetros por defecto.
4. **Potencial de mejora con tuning** — modelos que ya son competitivos pero pueden escalar.


In [ ]:
# ======================================================
# SELECCIÓN — Ajustar nombres según resultados reales
# ======================================================
selected = df_results.head(3)
discarded = df_results.iloc[3:]

print('=' * 70)
print('✅ MODELOS SELECCIONADOS para Sprint 4 (tuning):')
print('=' * 70)
for _, row in selected.iterrows():
    print(f"  🏆 {row['Modelo']:25s} | F1={row['F1']:.4f} | AUC={row['AUC-ROC']:.4f} | Gap={row['Overfit_Gap']:.4f}")

print()
print('❌ MODELOS DESCARTADOS:')
print('-' * 70)
for _, row in discarded.iterrows():
    overfit_note = ' (overfitting alto)' if row['Overfit_Gap'] > 0.10 else ''
    low_f1_note  = ' (F1 insuficiente)'  if row['F1'] < 0.80 else ''
    reason = overfit_note + low_f1_note or ' (superado por mejores candidatos)'
    print(f"  ✗  {row['Modelo']:25s} | F1={row['F1']:.4f}{reason}")

## 6. Documentar decisiones de selección

In [ ]:
# Registro de decisiones
decisions = []

for _, row in df_results.iterrows():
    selected_flag = row['Modelo'] in selected['Modelo'].values

    if selected_flag:
        reason = f"F1={row['F1']:.4f}, AUC={row['AUC-ROC']:.4f}. Seleccionado para tuning en Sprint 4."
    elif row['Overfit_Gap'] > 0.10:
        reason = f"Descartado: overfitting gap={row['Overfit_Gap']:.3f} > 0.10. Requiere regularización fuerte."
    elif row['F1'] < 0.80:
        reason = f"Descartado: F1={row['F1']:.4f} insuficiente. Rendimiento bajo baseline esperado."
    else:
        reason = f"Descartado: superado por los top 3 en F1 y AUC-ROC. No justifica el costo de tuning."

    decisions.append({
        'Modelo':   row['Modelo'],
        'F1':       row['F1'],
        'AUC-ROC':  row['AUC-ROC'],
        'Seleccionado': '✅ Sí' if selected_flag else '❌ No',
        'Justificación': reason,
    })

df_decisions = pd.DataFrame(decisions)
df_decisions.to_csv('../models/selection_decisions.csv', index=False)
print('✅ Decisiones guardadas en models/selection_decisions.csv')
df_decisions

## 7. Análisis de trade-offs

Para cada candidato seleccionado se analiza el trade-off **Precision vs. Recall**
y la velocidad estimada de entrenamiento.

In [ ]:
# Scatter: Precision vs Recall para todos los modelos
fig, ax = plt.subplots(figsize=(8, 6))

for _, row in df_results.iterrows():
    is_sel = row['Modelo'] in selected['Modelo'].values
    color  = '#1565c0' if is_sel else '#9e9e9e'
    size   = 180 if is_sel else 80
    marker = '*' if is_sel else 'o'
    ax.scatter(row['Recall'], row['Precision'], s=size, color=color,
               marker=marker, zorder=3, alpha=0.9)
    ax.annotate(row['Modelo'], (row['Recall'], row['Precision']),
                textcoords='offset points', xytext=(6, 4), fontsize=7.5)

ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Trade-off Precision vs. Recall — Baselines', fontsize=13, fontweight='bold')
ax.set_xlim(0.5, 1.02)
ax.set_ylim(0.5, 1.02)
ax.axhline(0.80, color='gray', linestyle='--', alpha=0.4)
ax.axvline(0.80, color='gray', linestyle='--', alpha=0.4)

# Leyenda manual
selected_patch  = mpatches.Patch(color='#1565c0', label='Seleccionado (★)')
discarded_patch = mpatches.Patch(color='#9e9e9e', label='Descartado')
ax.legend(handles=[selected_patch, discarded_patch], fontsize=9)

plt.tight_layout()
plt.savefig('../models/precision_recall_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Resumen ejecutivo — PB-12

### Modelos seleccionados para Sprint 4

In [ ]:
print('╔══════════════════════════════════════════════════════════════╗')
print('║          SPRINT 3 — SELECCIÓN FINAL DE CANDIDATOS           ║')
print('╠══════════════════════════════════════════════════════════════╣')
for rank, (_, row) in enumerate(selected.iterrows(), 1):
    print(f'║  #{rank} {row["Modelo"]:25s} F1={row["F1"]:.4f} AUC={row["AUC-ROC"]:.4f}  ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  → Sprint 4: GridSearchCV/RandomizedSearchCV para tuning    ║')
print('║  → Sprint 4: Ensambles (Bagging, Stacking) sobre candidatos ║')
print('╚══════════════════════════════════════════════════════════════╝')

## 9. Archivos generados — Sprint 3 completo

| Archivo | Generado en | Descripción |
|---------|-------------|-------------|
| `models/baseline_*.pkl` | 09 | 7 pipelines completos entrenados |
| `models/baseline_comparison.csv` | 09 | Tabla comparativa de métricas CV |
| `models/experiments_log.csv` | 09 | Registro de experimentos |
| `models/confusion_matrices.png` | 10 | Matrices de confusión |
| `models/roc_curves.png` | 10 | Curvas ROC comparativas |
| `models/precision_recall_curves.png` | 10 | Curvas Precision-Recall |
| `models/model_comparison_charts.png` | 11 | Barras agrupadas + overfitting |
| `models/radar_top3.png` | 11 | Radar chart top 3 |
| `models/precision_recall_scatter.png` | 11 | Scatter Precision vs Recall |
| `models/selection_decisions.csv` | 11 | Justificación de selección |

### Definition of Done — Sprint 3 ✅

- [x] Al menos 5 modelos baseline entrenados con parámetros por defecto.
- [x] Cross-validation Stratified (5-fold) aplicada a todos los modelos.
- [x] Tabla comparativa con métricas: Accuracy, F1, AUC-ROC, Precision, Recall.
- [x] Top 2–3 candidatos seleccionados con justificación documentada.
- [x] Modelos persistidos con joblib.
- [x] Registro de experimentos en CSV.
- [x] Código encapsulado en `src/models.py`.
